In [0]:
DEFAULT_AS_OF = spark.sql(
    "SELECT date_format(from_utc_timestamp(current_timestamp(), 'America/Toronto'), 'yyyy-MM-dd') AS d"
).first()["d"]

dbutils.widgets.text("catalog", "serverless_nv91sa_catalog")
dbutils.widgets.text("team", "team1")
dbutils.widgets.text("lakebase_host", "")
dbutils.widgets.text("lakebase_database", "")
dbutils.widgets.text("lakebase_user", "")
dbutils.widgets.text("lakebase_password", "")
dbutils.widgets.text("as_of_date", DEFAULT_AS_OF)

catalog = dbutils.widgets.get("catalog")
team = dbutils.widgets.get("team")

row_count = -1
try:
    import psycopg

    with psycopg.connect(
        host=dbutils.widgets.get("lakebase_host"),
        dbname=dbutils.widgets.get("lakebase_database"),
        user=dbutils.widgets.get("lakebase_user"),
        password=dbutils.widgets.get("lakebase_password"),
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT COUNT(*) FROM agent_memory WHERE team = %s", (team,))
            row_count = cursor.fetchone()[0]
except Exception as exc:
    # TODO: if psycopg is unavailable on the selected compute, install or switch to the preferred Lakebase client for the workshop runtime.
    print(f"Lakebase check skipped: {exc}")

status = "PASS" if row_count > 0 else "FAIL"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.ops.milestones (team STRING, round INT, status STRING, metric_name STRING, metric_value DOUBLE, ts TIMESTAMP)")
spark.sql(
    f"INSERT INTO {catalog}.ops.milestones VALUES ('{team}', 3, '{status}', 'agent_memory_rows', {float(row_count)}, current_timestamp())"
)

print({"status": status, "agent_memory_rows": row_count})
